# Monitoring Topological Invariants in Neural Operators using PINA

## What is Topology?
Topology studies properties of shapes that remain unchanged under continuous deformation—stretching, twisting, or bending, but not tearing or gluing. Unlike geometry, which cares about distances and angles, topology focuses on **connectivity** and **holes**.

## Betti Numbers (The "Shape Counters")

| Betti Number | What It Counts | Physics Meaning |
|--------------|----------------|-----------------|
| **β₀** | Number of connected components | Number of separate islands or bubbles |
| **β₁** | Number of 1‑dimensional holes/loops | Number of vortex rings or tunnels |
| **β₂** | Number of 2‑dimensional voids/cavities | Number of fully enclosed empty pockets |

For a 2D pressure field, we care about:
- **β₀ = 1** → The field is a single connected blob.
- **β₀ > 1** → There are disconnected "ghost islands."
- **β₁ = 0** → No spurious vortex rings.

## Why Monitor Topology?
Fourier Neural Operators (FNOs) are spectrally accurate but topologically blind. By truncating high frequencies (`n_modes`), they lose sharp boundaries, creating non‑physical disconnected components. Standard MSE loss cannot catch this—pixel‑wise values may match, but the shape is wrong. The `TopologyMonitor` fills this gap.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy import io
from torch.utils.data import DataLoader, TensorDataset
import warnings


warnings.filterwarnings("ignore", category=UserWarning, module="pina._src.model.fourier_neural_operator")

from pina import LabelTensor
from pina.model import FNO, FeedForward
from pina.solver import SupervisedSingleModelSolver
from pina.problem.zoo import SupervisedProblem
from pina import Trainer
from pina.topology import TopologyMonitor

torch.manual_seed(42)
np.random.seed(42)

print("✅ All imports loaded successfully.")

✅ All imports loaded successfully.


## 1. Generate Synthetic Darcy Data

In [2]:
def generate_synthetic_darcy(n_samples=100, grid_size=20):
    x = torch.linspace(0, 1, grid_size)
    y = torch.linspace(0, 1, grid_size)
    X, Y = torch.meshgrid(x, y, indexing='ij')
    inputs, outputs = [], []
    for _ in range(n_samples):
        k = 0.5 + 0.5 * torch.exp(-((X - 0.5)**2 + (Y - 0.5)**2) / 0.1)
        k = k + 0.05 * torch.randn(grid_size, grid_size)
        u = 1.0 - k
        inputs.append(k.unsqueeze(-1))
        outputs.append(u.unsqueeze(-1))
    return torch.stack(inputs), torch.stack(outputs)

k_train, u_train = generate_synthetic_darcy(n_samples=100, grid_size=20)
n_samples = 100
train_split = 80
k_sub = k_train[:n_samples]
u_sub = u_train[:n_samples]
train_input = k_sub[:train_split]
train_output = u_sub[:train_split]
val_input = k_sub[train_split:]
val_output = u_sub[train_split:]
print(f"Train: {len(train_input)}, Validation: {len(val_input)}")
print(f"Grid size: {k_train.shape[1]}x{k_train.shape[2]}")

Train: 80, Validation: 20
Grid size: 20x20


## 2. Build Tiny FNO

In [3]:
train_input_lt = LabelTensor(train_input, ['x'])
train_output_lt = LabelTensor(train_output, ['u'])
problem = SupervisedProblem(input_=train_input_lt, output_=train_output_lt)
problem.input_variables = ['x']
problem.output_variables = ['u']

lifting_net = FeedForward(input_dimensions=1, output_dimensions=8, layers=[8, 8])
projecting_net = FeedForward(input_dimensions=8, output_dimensions=1, layers=[8, 8])
fno = FNO(
    lifting_net=lifting_net,
    projecting_net=projecting_net,
    n_modes=4,
    dimensions=2,
    n_layers=1,
    padding=2,
    inner_size=8,
)
solver = SupervisedSingleModelSolver(problem=problem, model=fno)
print("✅ Model built.")


✅ Model built.


## 3. DataLoaders & TopologyMonitor

In [4]:
def pina_collate(batch):
    inputs = torch.stack([item[0] for item in batch])
    outputs = torch.stack([item[1] for item in batch])
    data_dict = {'input': LabelTensor(inputs, ['x']), 'target': LabelTensor(outputs, ['u'])}
    return [('data', data_dict)]

train_loader = DataLoader(TensorDataset(train_input, train_output), batch_size=8, shuffle=True, collate_fn=pina_collate)
val_loader = DataLoader(TensorDataset(val_input, val_output), batch_size=8, shuffle=False, collate_fn=pina_collate)

# FIX: Extract signature must accept 3 arguments (batch, batch_idx, outputs)
def custom_extractor(batch, batch_idx, outputs):
    try:
        return batch[0][1]['input']
    except (IndexError, KeyError, TypeError):
        return None

monitor = TopologyMonitor(
    expected_beta_0=1,
    expected_beta_1=0,
    monitor_freq=1,
    warmup_epochs=0,
    mode="warn",
    min_persistence=0.5,
    collect_predictions=custom_extractor,
)
print("✅ Monitor attached.")

✅ Monitor attached.


## 4. Train (Single Epoch)

In [5]:
trainer = Trainer(solver=solver, max_epochs=1, callbacks=[monitor], accelerator="cpu")
trainer.fit(solver, train_dataloaders=train_loader, val_dataloaders=val_loader)
print("✅ Training complete.")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.

  | Name         | Type       | Params | Mode  | FLOPs
------------------------------------------------------------
0 | _pina_models | ModuleList | 2.4 K  | train | 0    
1 | _loss_fn     | MSELoss    | 0      | train | 0    
------------------------------------------------------------
2.4 K     Trainable params
0         Non-trainable params
2.4 K     Total params
0.010     Total estimated model params size (MB)
23        Modules in train mode
0         Modules in eval mode
0        

Sanity Checking: |                                          | 0/? [00:00<?, ?it/s]

/home/vrinda/miniconda3/envs/pina-dev/lib/python3.10/site-packages/lightning/pytorch/utilities/_pytree.py: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/vrinda/miniconda3/envs/pina-dev/lib/python3.10/site-packages/lightning/pytorch/trainer/connectors/data_connector.py: PossibleUserWarning: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/home/vrinda/PINA/pina/_src/model/fourier_neural_operator.py: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /__w/pytorch/pytorch/torch/csrc/aut

Training: |                                                 | 0/? [00:00<?, ?it/s]

/home/vrinda/PINA/pina/_src/model/fourier_neural_operator.py: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:349.)
/home/vrinda/PINA/pina/_src/model/fourier_neural_operator.py: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:349.)
/home/vrinda/PINA/pina/_src/model/fourier_neural_operator.py: UserWarning: Using a n

Validation: |                                               | 0/? [00:00<?, ?it/s]

/home/vrinda/PINA/pina/_src/model/fourier_neural_operator.py: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:349.)
/home/vrinda/PINA/pina/_src/model/fourier_neural_operator.py: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/python_variable_indexing.cpp:349.)
/home/vrinda/PINA/pina/_src/model/fourier_neural_operator.py: UserWarning: Using a n

✅ Training complete.


## 5. Visualise Prediction & Betti Numbers

In [ ]:
from pina._src.callback.topology.profiler import TopologicalProfiler

sample_input = val_input[0:1]
sample_gt = val_output[0]
with torch.no_grad():
    pred = fno(sample_input)

profiler = TopologicalProfiler(min_persistence=0.5)
pred_channels_first = pred.permute(0, 3, 1, 2)
result = profiler.compute(pred_channels_first)

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(sample_gt.squeeze().numpy(), cmap='jet')
ax[0].set_title("Ground Truth")
ax[0].axis('off')
ax[1].imshow(pred[0, ..., 0].numpy(), cmap='jet')
ax[1].set_title(f"Prediction\nβ₀ = {result.beta_0_mean:.0f}, β₁ = {result.beta_1_mean:.0f}")
ax[1].axis('off')
plt.tight_layout()
plt.show()
print(f"Topological Signature: β₀ = {result.beta_0_mean:.2f}, β₁ = {result.beta_1_mean:.2f}")

✅ TopologyMonitor is ready!


## Interpreting the Results
The ground truth is a single connected pressure field (β₀ = 1). If the FNO prediction shows disconnected islands, β₀ will be > 1 and the `TopologyMonitor` will raise a warning.

## Limitations
1. **Computational Cost**: Persistent homology runs on CPU; large grids (>512²) add overhead.
2. **GUDHI Dependency**: Requires `pip install gudhi`.
3. **Interpretability**: Betti numbers are summary statistics; they indicate *that* an anomaly exists, but not *where*.